# 🦆 Local-First ELT Pipeline: Retail Sales Star Schema with DuckDB

## Arsitektur
1. **Ingestion:** Mengunduh dataset Retail Sales dari Kaggle menggunakan `kagglehub`.
2. **Data Lake:** Mengonversi CSV mentah ke format **Parquet** (columnar storage) untuk performa optimal.
3. **ELT Engine:** **DuckDB** membaca Parquet dan melakukan transformasi SQL.
4. **Data Modeling:** Membangun **Star Schema** (`fact_sales`, `dim_customer`, `dim_product`) dengan *feature engineering* (misal: `age_group`).

## Tech Stack
- **Database:** DuckDB (In-process OLAP)
- **Storage:** Apache Parquet
- **Language:** Python + SQL
- **Concept:** ELT, Star Schema, Data Lakehouse, Feature Engineering

In [ ]:
# Install library yang dibutuhkan
!pip install kagglehub duckdb pandas -q

import kagglehub
import duckdb
import pandas as pd
import os

# 1. Download dataset dari Kaggle secara otomatis
print("⏳ Mengunduh dataset dari Kaggle...")
path = kagglehub.dataset_download("mohammadtalib786/retail-sales-dataset")
print(f"✅ Dataset tersimpan di: {path}")

# 2. Cari file CSV di dalam folder yang diunduh
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
csv_path = os.path.join(path, csv_files[0])
print(f"📄 File CSV ditemukan: {csv_files[0]}")

# 3. Inisialisasi koneksi DuckDB
con = duckdb.connect('retail_warehouse.db')
print("✅ DuckDB terhubung.")

⏳ Mengunduh dataset dari Kaggle...
Using Colab cache for faster access to the 'retail-sales-dataset' dataset.
✅ Dataset tersimpan di: /kaggle/input/retail-sales-dataset
📄 File CSV ditemukan: retail_sales_dataset.csv
✅ DuckDB terhubung.


In [ ]:
parquet_path = 'raw_retail_sales.parquet'

# Baca CSV langsung dan simpan sebagai Parquet (Simulasi Data Lake Ingestion)
con.sql(f"""
    COPY (
        SELECT * FROM read_csv_auto('{csv_path}')
    ) TO '{parquet_path}' (FORMAT PARQUET)
""")

print(f"✅ Data Lake siap: {parquet_path}")
print("📊 Preview 5 baris data mentah dari Parquet:")
con.sql(f"SELECT * FROM read_parquet('{parquet_path}') LIMIT 5").df()

✅ Data Lake siap: raw_retail_sales.parquet
📊 Preview 5 baris data mentah dari Parquet:


,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [ ]:
# Buat schema untuk hasil analitik
con.sql("CREATE SCHEMA IF NOT EXISTS analytics")

# 1. DIMENSION TABLE: dim_customer
# Dilengkapi Feature Engineering: membuat kategori 'age_group' dari kolom 'Age'
con.sql("""
    CREATE OR REPLACE TABLE analytics.dim_customer AS
    SELECT DISTINCT
        "Customer ID" AS customer_id,
        Gender AS gender,
        Age AS age,
        CASE
            WHEN Age < 25 THEN '18-24'
            WHEN Age < 35 THEN '25-34'
            WHEN Age < 50 THEN '35-49'
            ELSE '50+'
        END AS age_group
    FROM read_parquet('raw_retail_sales.parquet')
""")

# 2. DIMENSION TABLE: dim_product
# Mengagregasi data untuk membuat master data produk
con.sql("""
    CREATE OR REPLACE TABLE analytics.dim_product AS
    SELECT
        "Product Category" AS product_category,
        ROUND(AVG("Price per Unit"), 2) AS avg_price_per_unit,
        COUNT(DISTINCT "Transaction ID") AS total_transactions
    FROM read_parquet('raw_retail_sales.parquet')
    GROUP BY "Product Category"
""")

# 3. FACT TABLE: fact_sales
# Tabel fakta berisi transaksi yang menghubungkan ke dimensi
con.sql("""
    CREATE OR REPLACE TABLE analytics.fact_sales AS
    SELECT
        "Transaction ID" AS transaction_id,
        CAST("Date" AS DATE) AS sale_date,
        "Customer ID" AS customer_id,
        "Product Category" AS product_category,
        Quantity AS quantity,
        "Price per Unit" AS price_per_unit,
        "Total Amount" AS total_amount
    FROM read_parquet('raw_retail_sales.parquet')
""")

print("✅ Star Schema berhasil dibuat!")
print("\n📋 Tabel yang tersedia di schema 'analytics':")
con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'analytics'
""").df()

✅ Star Schema berhasil dibuat!

📋 Tabel yang tersedia di schema 'analytics':


,table_name
0,dim_customer
1,dim_product
2,fact_sales


In [ ]:
# Query 1: Total Revenue & Item Terjual per Kategori Produk
print("💰 Total Revenue per Product Category:")
revenue_by_category = con.sql("""
    SELECT
        p.product_category,
        SUM(f.total_amount) AS total_revenue,
        SUM(f.quantity) AS total_items_sold
    FROM analytics.fact_sales f
    JOIN analytics.dim_product p ON f.product_category = p.product_category
    GROUP BY p.product_category
    ORDER BY total_revenue DESC
""").df()
print(revenue_by_category)

# Query 2: Purchasing Behavior berdasarkan Demografi (Age Group & Gender)
print("\n👥 Purchasing Behavior by Age Group & Gender:")
demographic_sales = con.sql("""
    SELECT
        c.age_group,
        c.gender,
        COUNT(DISTINCT f.transaction_id) AS total_transactions,
        ROUND(SUM(f.total_amount), 2) AS total_revenue
    FROM analytics.fact_sales f
    JOIN analytics.dim_customer c ON f.customer_id = c.customer_id
    GROUP BY c.age_group, c.gender
    ORDER BY c.age_group, total_revenue DESC
""").df()
print(demographic_sales)

# Tutup koneksi
con.close()
print("\n✅ Pipeline selesai! Database tersimpan di: retail_warehouse.db")

💰 Total Revenue per Product Category:
  product_category  total_revenue  total_items_sold
0      Electronics       156905.0             849.0
1         Clothing       155580.0             894.0
2           Beauty       143515.0             771.0

👥 Purchasing Behavior by Age Group & Gender:
  age_group  gender  total_transactions  total_revenue
0     18-24    Male                  77        38730.0
1     18-24  Female                  72        35920.0
2     25-34  Female                 102        51850.0
3     25-34    Male                 101        45240.0
4     35-49  Female                 169        73305.0
5     35-49    Male                 143        67800.0
6       50+  Female                 167        71765.0
7       50+    Male                 169        71390.0

✅ Pipeline selesai! Database tersimpan di: retail_warehouse.db
